# Acceptance: the Kiribati port gaps

**This is a sign-off page, not a tutorial.** It checks roadmap step 52 (`feat/kiribati-gaps`,
plan `plans/kiribati-gaps.plan.md`): the small summer4 gaps that porting the Kiribati TB model
found, each fixed with the spelling the port wanted to write.

| Section | Gap | Claim |
|---|---|---|
| A | 1 (`CX51`) | `Lookup` indexes a table computed from the parameters, built once per run |
| B | 5, 6 (`CX54`) | `TraitChain` moves between paired states, with rate-expression rates for a split |
| C | 4 (`CX53`) | each request can name its own strata, including a whole-name template |
| D | 7 (`CX55`) | a `GroupedOutput` of a plain `Capture` keeps its property map |
| E | 2 (`CX52`) | a `jnp` function on a grouped `defer` argument gives a clear error; NumPy works |
| F | W4 (`CX56`) | `differences(sign=-1)` is averted burden, quantiled per draw |
| G | W7 (`CX58`) | `wf.Optax(optax.lbfgs())` runs under `wf.optimize` |

The large gaps (an exact running total of a flow, forward-mode calibration, other start maps for
`optimize`, `MCMCRun` save/load, a joint coordinate map) are recorded as `CX57` and `CX59`-`CX62`
with `futureplans/` notes; nothing here checks them.

In [ ]:
from typing import Any, NamedTuple

import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    Dest,
    FlowModel,
    GroupedOutput,
    Lookup,
    Multiply,
    OutputSet,
    Param,
    Params,
    Property,
    PropertyData,
    PropertyMap,
    Reduce,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    Time,
    TraitChain,
    TransitionFlow,
    defer,
    field_refs,
    floor,
)
from summer4.data import Data
from summer4.epi.calibration import BayesianModel, NormalLikelihood, Uniform
from summer4.epi.calibration import workflow as wf
from summer4.epi.calibration.posterior_runs import PosteriorRuns
from summer4.flows.rates import Capture

state = Property("state", ("S", "I", "R"))
age = Property("age", ("young", "old"))
pmap = PropertyMap.from_property(state).stratify(age)
y0 = np.array([600.0, 380.0, 15.0, 5.0, 0.0, 0.0])
RUN = dict(t0=0.0, t1=8.0, dt=0.05, solver="euler")


def sir(rate: Any, recovery: Any = 0.5) -> Any:
    model = FlowModel(pmap)
    model.add_flow(TransitionFlow("infection", state["S"], state["I"], rate))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], recovery))
    return model.compile()

## A. A yearly table computed from the parameters, indexed by `Lookup`

The Kiribati mixing matrix changes every year, and the yearly stack is computed from the
parameters. Before this step `Lookup` read its table only from a parameter, so the port either
computed the stack in `prepare_fn` or picked the year's row with a second `defer`. Now the table is
any rate expression. Here a yearly contact rate, `base * exp(-decay * year)`, is computed by one
`defer` of the parameters and read by `Lookup(stacks.yearly, floor(Time()))`. The second route
computes the same stack on the host and passes it as a parameter. The figure draws the infectious
population both ways, and the contact rate saved during the solve.

In [ ]:
class Stacks(NamedTuple):
    yearly: Any


def build_stack(params: Any) -> dict[str, Any]:
    years = jnp.arange(10.0)
    return {"yearly": params["base"] * jnp.exp(-params["decay"] * years)}


params_a = {"base": 0.002, "decay": 0.4}
stacks = field_refs(defer(build_stack)(Params()), Stacks)
contact = Lookup(stacks.yearly, floor(Time()))
computed = sir(contact)
as_param = sir(Lookup(Param("stack"), floor(Time())))

out_a = OutputSet()
out_a["infectious"] = Compartments().select(state["I"]).sum_by()
out_a["contact"] = contact
named_computed = computed.run(params_a, y0, outputs=out_a, **RUN)
stack = np.asarray(build_stack(params_a)["yearly"])
out_param = OutputSet()
out_param["infectious"] = Compartments().select(state["I"]).sum_by()
named_param = as_param.run({"stack": stack}, y0, outputs=out_param, **RUN)

times_a = np.asarray(named_computed["infectious"].times.values)
pd.DataFrame(
    {
        "I: Lookup(computed stack)": np.asarray(named_computed["infectious"].array()),
        "I: Lookup(Param('stack'))": np.asarray(named_param["infectious"].array()),
        "contact rate x 1e5 (saved)": 1e5 * np.asarray(named_computed["contact"].array()),
    },
    index=pd.Index(times_a, name="year"),
).plot(title="A. Yearly table computed from the parameters", labels={"value": "", "variable": ""})

In [ ]:
# The table's node is in the hoist table: it is evaluated once per run, not per step.
pd.DataFrame(
    [(type(e.node).__name__, getattr(e.node, "key", ""), e.part) for e in computed.hoist_table.entries],
    columns=["node", "key", "part"],
)

> **What to check (A).** The two infectious curves lie on each other. The saved contact rate steps
> down once a year (200, 134, 90, … per 100,000). The hoist table lists one `Item` node with key
> `yearly`: the stack is built once per run.

## B. `TraitChain` within the state property, with rate-expression rates

`TraitChain` is any explicit list of `(source, dest)` pairs within one property, not only ageing.
The clean Kiribati port uses it for paired state moves. Progression from `incipient` splits
between a low-infectious and an infectious state in proportions `1 - p` and `p`. Before this step
the per-pair `rates` had to be numbers, so the port wrote two `Dest` adjustments. Now
`rates=(1 - p, p)`. The figure draws the two destination states both ways.

In [ ]:
tb = Property("tb", ("incipient", "low", "high", "cleared"))
tb_map = PropertyMap.from_property(tb).stratify(age)
tb_y0 = np.array([100.0, 80.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
SPLIT = (("incipient", "low"), ("incipient", "high"))
p = Param("p")
params_b = {"rate": 0.4, "p": 0.3}


def tb_run(progression: TransitionFlow) -> Any:
    model = FlowModel(tb_map)
    model.add_flow(progression)
    model.add_flow(TransitionFlow("clearance", tb["low"], tb["cleared"], 0.2))
    out = OutputSet()
    out.compartments("tb", by=tb, naming="{tb}")
    return model.compile().run(params_b, tb_y0, outputs=out, **RUN)


with_rates = tb_run(
    TransitionFlow(
        "progression", tb.present(), tb.present(), Param("rate"),
        pairing=TraitChain(tb, SPLIT, rates=(1 - p, p)),
    )
)
by_dest = tb_run(
    TransitionFlow(
        "progression", tb.present(), tb.present(), Param("rate"),
        pairing=TraitChain(tb, SPLIT),
        adjust=[Multiply(1 - p, where=Dest(tb["low"])), Multiply(p, where=Dest(tb["high"]))],
    )
)
frame_b = {}
for trait in ("low", "high"):
    frame_b[f"{trait}: rates=(1 - p, p)"] = np.asarray(with_rates[trait].array())
    frame_b[f"{trait}: Dest adjustments"] = np.asarray(by_dest[trait].array())
pd.DataFrame(frame_b, index=pd.Index(times_a, name="year")).plot(
    title="B. A split from one source state, two spellings",
    labels={"value": "people", "variable": ""},
)

> **What to check (B).** Each state's two lines lie on each other. `high` has no exit and rises
> towards 0.3 × 180 = 54 (about 52 by year 8); `low` peaks near 63 and then falls as it clears. `help(TraitChain)`
> shows the paired-state and split examples.

## C. Each request names its own strata

summer2-port names come in more than one style. The default `X{prop}_{trait}` suffix names
`recoveredXage_young`; the Kiribati port also has names like
`all_cause_mortality_from_<state>_age_<age>`. Before this step the set had one `naming`, so the
second style needed a loop. Now `naming=` goes on the request, and a whole-name template
(`{name}` plus property names) is formatted once per stratum. The table lists the names one
request added; the figure draws a default-named and a template-named output of the same stratum.

In [ ]:
reach = Property("reach", ("near", "far"))
pmap_c = pmap.stratify(reach)
model_c = FlowModel(pmap_c)
model_c.add_flow(TransitionFlow("infection", state["S"], state["I"], 0.3))
model_c.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.5))
y0_c = PropertyData.wrap(pmap_c, np.arange(1.0, pmap_c.size + 1.0) * 10)

out_c = OutputSet()
out_c.flow("recovered", "recovery", by=[(age, reach)])
out_c.flow("recovery", "recovery", by=[(age, reach)], naming="all_{name}_from_{age}_reach_{reach}")
named_c = model_c.compile().run({}, y0_c, outputs=out_c, **RUN)
pd.Series([n for n in out_c.kept() if n.startswith("all_")], name="names added by one request")

In [ ]:
pd.DataFrame(
    {
        "recoveredXage_oldXreach_far (set naming)": np.asarray(
            named_c["recoveredXage_oldXreach_far"].array()
        ),
        "all_recovery_from_old_reach_far (request naming)": np.asarray(
            named_c["all_recovery_from_old_reach_far"].array()
        ),
    },
    index=pd.Index(times_a, name="year"),
).plot(title="C. One stratum, two naming styles", labels={"value": "per year", "variable": ""})

> **What to check (C).** The table lists four names, `all_recovery_from_<age>_reach_<reach>`, one
> per age and reachability. The two lines in the figure lie on each other.

## D. A `GroupedOutput` of a plain `Capture` keeps its property map

Gate notebook 40 found that saving an explicit `Capture(name, inner)` with `GroupedOutput(name)`
returned a raw array, so `.select(age[...])` failed; only `ForceOfInfection`'s own capture was
labelled. Now a plain `Capture` records the grouping of its inner expression. Here the infection
rate reads the population by age through a capture, and the figure selects each age from the
captured output and from the same expression saved directly.

In [ ]:
by_age = Capture("by_age", Reduce(age))
cm_d = sir(Param("b") * by_age / 1000.0)
out_d = OutputSet()
out_d["captured"] = GroupedOutput("by_age")
out_d["saved"] = Reduce(age)
named_d = cm_d.run({"b": 0.4}, y0, outputs=out_d, **RUN)
print("dims:", named_d["captured"].dims)
frame_d = {}
for a in age.traits:
    frame_d[f"{a}: GroupedOutput('by_age').select"] = np.asarray(
        named_d["captured"].select(age[a]).sum_by().array()
    )
    frame_d[f"{a}: outputs['saved'] = Reduce(age)"] = np.asarray(
        named_d["saved"].select(age[a]).sum_by().array()
    )
pd.DataFrame(frame_d, index=pd.Index(times_a, name="year")).plot(
    title="D. Captured population by age", labels={"value": "people", "variable": ""}
)

> **What to check (D).** `dims` prints `('time', 'age')`. For each age the two lines lie on each
> other, and both stay constant (the model has no births or deaths): 615 young and 385 old.

## E. `jnp` on a grouped `defer` argument: a clear error, and the NumPy spelling

A per-age table reaches a `defer` function as a `GroupedRate`. NumPy functions work on it and
keep the grouping. `jax.numpy` functions do not: summer4 deliberately does not teach JAX to unwrap
it, because JAX's own operators would then drop the grouping silently in the middle of an
expression (`p * g` with `p` a traced parameter). Before this step the `jnp` call failed deep in
JAX. Now the error names the function, the argument and both fixes. The figure draws the NumPy
spelling against the same rate written as a rate expression.

In [ ]:
deaths = Data.table(np.array([0.0, 8.0]), np.array([[0.02, 0.01], [0.04, 0.03]]), over=age)


def on_treatment_np(mu: Any, duration: Any) -> Any:
    return 1.0 - np.exp(-duration * mu)


def on_treatment_jnp(mu: Any, duration: Any) -> Any:
    return 1.0 - jnp.exp(-duration * mu)


try:
    sir(defer(on_treatment_jnp)(deaths.interp(), Param("duration"))).run(
        {"duration": 2.0}, y0, **RUN
    )
except TypeError as err:
    print(err)

In [ ]:
out_e = OutputSet()
out_e["infectious"] = Compartments().select(state["I"]).sum_by()
np_route = sir(defer(on_treatment_np)(deaths.interp(), Param("duration")))
expr_route = sir(1.0 - np.exp(-Param("duration") * deaths.interp()))
pd.DataFrame(
    {
        "defer(on_treatment_np)": np.asarray(
            np_route.run({"duration": 2.0}, y0, outputs=out_e, **RUN)["infectious"].array()
        ),
        "rate expression": np.asarray(
            expr_route.run({"duration": 2.0}, y0, outputs=out_e, **RUN)["infectious"].array()
        ),
    },
    index=pd.Index(times_a, name="year"),
).plot(title="E. NumPy inside defer keeps the grouping", labels={"value": "infectious", "variable": ""})

> **What to check (E).** The printed error starts `defer(on_treatment_jnp) received a GroupedRate
> (argument 0)`, says to use `np.exp` or `g.data`, and quotes JAX's own message on its first line.
> The two lines in the figure lie on each other.

## F. Averted burden: `differences(sign=-1)`

`PosteriorRuns.differences` gave quantiles of `scenario - ref`. The Kiribati results report
averted burden, `ref - scenario`, and quantiles do not commute with negation, so the frame cannot
be flipped afterwards; the port recomputed it from the draws. Here 400 synthetic draws of deaths
under a reference and a screening scenario are summarised three ways: `sign=-1`, the quantiles of
the averted draws computed by hand, and the `sign=1` frame negated.

In [ ]:
rng = np.random.default_rng(3)
ref_draws = rng.gamma(4.0, 25.0, size=(400, 1))
scen_draws = ref_draws * rng.uniform(0.3, 1.1, size=(400, 1))
runs = PosteriorRuns(
    times=np.array([2030.0]),
    series={"baseline": {"deaths": ref_draws}, "screening": {"deaths": scen_draws}},
    n=400,
    scenario_names=("baseline", "screening"),
    output_names=("deaths",),
    draws={},
)
q = (0.025, 0.25, 0.5, 0.75, 0.975)
averted = runs.differences("baseline", {"d": "deaths"}, at=2030.0, q=q, sign=-1)["screening"]
plain = runs.differences("baseline", {"d": "deaths"}, at=2030.0, q=q)["screening"]
frame_f = pd.DataFrame(
    {
        "differences(sign=-1)": averted["d"].to_numpy(),
        "np.quantile(ref - scenario)": np.quantile(ref_draws[:, 0] - scen_draws[:, 0], q),
        "-differences(sign=1) (wrong)": -plain["d"].to_numpy(),
    },
    index=pd.Index([str(v) for v in q], name="quantile"),
)
frame_f.plot.bar(
    barmode="group", title="F. Averted deaths, three ways", labels={"value": "deaths averted"}
)

> **What to check (F).** The first two bars agree at every quantile. The third (the negated
> `sign=1` frame) is the same values in reverse order: its 2.5 % bar is the averted 97.5 % bar,
> so it is wrong as a quantile table.

## G. `wf.Optax(optax.lbfgs())` runs under `wf.optimize`

`wf.Optax` called `update(grads, state, z, value=loss)` without the `value_fn` that the L-BFGS
zoom line search needs, so the optimiser the Kiribati port wanted could not run. Now each step
passes `value=`, `grad=` and `value_fn=`. A one-parameter SIR calibration is fitted from four
starts; the figure draws each start's fitted infection rate against the value that made the data.

In [ ]:
st = Property("state", ("S", "I", "R"))
pm = PropertyMap.from_property(st)
fm = FlowModel(pm)
fm.add_flow(TransitionFlow("infection", st["S"], st["I"], Param("infection")))
fm.add_flow(TransitionFlow("recovery", st["I"], st["R"], 0.1))
cm_g = fm.compile()
y0_g = PropertyData.wrap(pm, np.array([999.0, 1.0, 0.0]))
times_g = np.array([0.0, 20.0, 40.0, 60.0])
qty = Compartments(where=st["I"])
run_g = dict(t0=0.0, t1=80.0, dt=1.0, solver="euler")
truth = cm_g.run(
    {"infection": 0.35}, y0_g, save=SavePlan(requests={"I": SaveRequest(qty, ts=times_g)}), **run_g
)
observed = np.asarray(truth["I"].at_times(times_g).values.data).reshape(-1)
bm = BayesianModel(
    cm_g,
    {},
    priors=(Uniform("infection", 0.05, 1.0),),
    targets=TargetSet(
        targets=(
            Target(key="I", times=times_g, values=observed, quantity=qty,
                   likelihood=NormalLikelihood(sd=5.0)),
        )
    ),
    y0=y0_g,
    run_kwargs=run_g,
)
starts = wf.lhs(bm, 4, seed=1)
fit = wf.optimize(bm, starts, method=wf.Optax(optax.lbfgs()), chunk_steps=10, max_steps=30)
pd.DataFrame(
    {
        "start": np.asarray(starts.params["infection"]),
        "fitted (Optax(optax.lbfgs()))": np.asarray(fit.candidates.params["infection"]),
        "truth": np.full(4, 0.35),
    },
    index=pd.Index(range(4), name="start"),
).plot.bar(barmode="group", title="G. L-BFGS fits from four starts", labels={"value": "infection rate"})

> **What to check (G).** The four starts are spread over the prior (0.17 to 0.99); every fitted
> bar is the same, 0.355, within 0.01 of the truth bar (0.35).